# 🧪 Landing Page A/B Test: Which Version Should We Launch?

This project evaluates an **A/B test** run on the landing page of an e-commerce site. Between **January 1 and 28, 2026**, **40,000 users** were randomly shown either **Version A (control)** or **Version B (variant)**. The goal is to decide, with statistical evidence, **which version the business should roll out**.

---

### ❓ Business questions

1. 💰 Do buyers spend more on one version than on the other?
2. 🎯 Does one version convert a larger share of visitors into buyers?
3. 🌐 Is conversion related to the **traffic source**?
4. 👤 Is conversion related to the **user type** (new vs. returning)?
5. 💵 Which version generates more **revenue per visitor**, the metric that combines conversion and spend?

### ⚡ Key result at a glance

> **Version B wins.** Its conversion rate rises from **12.57% to 15.96%** (+3.38 percentage points) and its revenue per exposed user is **42.83% higher**, with a 95% confidence interval of **+35.30% to +50.79%**. **Recommendation: roll out Version B.**

### 📋 Contents

| Step | Topic |
|---|---|
| 1 | 🧩 Load and validate the data |
| 2 | 💰 Average spend per buyer (Welch's t-test) |
| 3 | 🎯 Conversion rate (two-proportion z-test) |
| 4 | 🌐 Traffic source vs. conversion (chi-square test) |
| 5 | 👤 User type vs. conversion (chi-square test) |
| 6 | 📊 Visualizing the categorical results |
| 7 | 💵 Revenue per exposed user (bootstrap) |
| 8 | 🧭 Executive summary for stakeholders |
| 9 | 💾 Export the clean dataset |

**🧰 Tools:** Python · pandas · NumPy · SciPy · statsmodels · Matplotlib

### 🗂️ Dataset description

The file `landing_experiment.csv` has one row per user. The original file uses a few Spanish column names and labels; in **Step 1.3** they are standardized to English.

| Column | Original name | Description |
|---|---|---|
| `user_id` | `user_id` | Unique user identifier |
| `date` | `date` | Date the user saw the landing page |
| `landing` | `landing` | Landing page version shown (`A` = control, `B` = variant) |
| `region` | `region` | Geographic region (North, Central, South, West, East) |
| `device` | `dispositivo` | Device used (Mobile, Desktop) |
| `traffic_source` | `traffic_source` | Acquisition channel (Organic, Ads, Email, Referral) |
| `user_type` | `user_type` | New or returning user |
| `converted` | `converted` | `1` if the user made a purchase, `0` otherwise |
| `spend` | `gasto` | Amount spent, in monetary units (`0` if the user did not buy) |

## 🧩 Step 1: Load and Validate the Data

### 1.1 Setup

In [ ]:
# ============================================================
# 1.1 SETUP: libraries, global settings and helper functions
# ============================================================
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
import statsmodels
from scipy import stats
from scipy.stats import chi2_contingency, chisquare, mannwhitneyu, norm, ttest_ind
from scipy.stats import t as student_t
from statsmodels.stats.proportion import proportions_ztest

try:
    from IPython.display import display
except ImportError:  # running outside Jupyter
    display = print

# ---------- Analysis settings ----------
ALPHA = 0.05      # significance level used in every test
N_BOOT = 10_000   # number of bootstrap resamples
SEED = 42         # fixed seed so the bootstrap results are reproducible

# ---------- Color palette ----------
COLOR_A = "#9AA5B1"              # Version A (control)
COLOR_B = "#2A9D8F"              # Version B (variant)
COLOR_NOT_CONVERTED = "#BFD7EA"
COLOR_CONVERTED = "#2A9D8F"
COLOR_ABOVE = "#2A9D8F"          # above the overall conversion rate
COLOR_BELOW = "#E76F51"          # below the overall conversion rate

# ---------- Plot style (compatible with old and new Matplotlib versions) ----------
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.linestyle": ":",
    "grid.alpha": 0.6,
    "axes.axisbelow": True,
})


# ---------- Helper functions ----------
def fmt_p(p):
    """Format a p-value, switching to scientific notation when it is very small."""
    return f"{p:.2e}" if p < 0.001 else f"{p:.4f}"


def decision(p, alpha=ALPHA):
    """Return a readable decision for a hypothesis test."""
    return "✅ Significant (reject H₀)" if p < alpha else "❌ Not significant (fail to reject H₀)"


def iqr_upper_limit(values):
    """Upper outlier limit according to the IQR rule: Q3 + 1.5 · IQR."""
    q1, q3 = np.percentile(values, [25, 75])
    return q3 + 1.5 * (q3 - q1)


def cramers_v_label(v):
    """Qualitative size of Cramér's V."""
    if v < 0.10:
        return "negligible 🔹"
    if v < 0.30:
        return "small 🔸"
    if v < 0.50:
        return "moderate 🟠"
    return "large 🔴"


print("✅ Environment ready")
print(f"🐍 pandas {pd.__version__} | NumPy {np.__version__} | SciPy {scipy.__version__} | "
      f"statsmodels {statsmodels.__version__} | Matplotlib {matplotlib.__version__}")

### 1.2 Load the data

The cell below looks for `landing_experiment.csv` next to the notebook (the clean export `landing_experiment_clean.csv` from Step 9 also works). **In Google Colab**, if neither file has been uploaded yet, an upload button appears: click it and select the CSV.

In [ ]:
# ============================================================
# 1.2 LOAD THE DATA
# ============================================================
DATA_FILE = "landing_experiment.csv"
CLEAN_FALLBACK = "landing_experiment_clean.csv"   # the clean export (Step 9) also works as input

# Look for the file in the usual locations (local folder, Colab, original platform)
candidate_paths = [
    Path(folder) / name
    for name in (DATA_FILE, CLEAN_FALLBACK)
    for folder in (".", "/content", "data", "/datasets")
]
data_path = next((path for path in candidate_paths if path.exists()), None)

# In Google Colab, ask for the file if it is not there yet
if data_path is None:
    try:
        from google.colab import files
        print(f"📤 '{DATA_FILE}' was not found. Select it in the upload box below.")
        uploaded = files.upload()
        data_path = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError(
            f"'{DATA_FILE}' was not found. Place it in the same folder as this notebook."
        )

df_raw = pd.read_csv(data_path)

print(f"✅ Data loaded from: {data_path}")
print(f"📐 Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

### 1.3 Standardize column names and labels to English

The raw file mixes Spanish and English. To keep the whole project in English, two columns are renamed and two categorical columns get English labels. The original data stays untouched in `df_raw`; the analysis uses the standardized copy `df`.

In [ ]:
# ============================================================
# 1.3 STANDARDIZE COLUMN NAMES AND CATEGORY LABELS
# ============================================================
COLUMN_NAMES = {"dispositivo": "device", "gasto": "spend"}

CATEGORY_LABELS = {
    "region": {"Norte": "North", "Centro": "Central", "Sur": "South",
               "Occidente": "West", "Oriente": "East"},
    "user_type": {"Nuevo": "New", "Recurrente": "Returning"},
}

COLUMN_ORDER = ["user_id", "date", "landing", "region", "device",
                "traffic_source", "user_type", "converted", "spend"]

df = df_raw.rename(columns=COLUMN_NAMES).copy()

missing_columns = set(COLUMN_ORDER) - set(df.columns)
if missing_columns:
    raise KeyError(f"Missing expected columns: {sorted(missing_columns)}")

for column, mapping in CATEGORY_LABELS.items():
    df[column] = df[column].replace(mapping)

df = df[COLUMN_ORDER]

print("🔤 Renamed columns")
for old, new in COLUMN_NAMES.items():
    status = "✅" if old in df_raw.columns else "(already in English)"
    print(f"   {old} → {new} {status}")

print("\n🌍 Translated labels")
for column, mapping in CATEGORY_LABELS.items():
    print(f"   {column}: " + ", ".join(f"{es} → {en}" for es, en in mapping.items()))

print(f"\n📋 Final columns: {list(df.columns)}")

### 1.4 First look at the data

In [ ]:
# First five rows
df.head()

In [ ]:
# Column types and non-null counts
df.info()

**🔎 Initial observations**

- The dataset has **40,000 rows and 9 columns**, and every column has 40,000 non-null values, so there are no missing values.
- `landing`, `region`, `device`, `traffic_source` and `user_type` are text columns, which fits their categorical content.
- `converted` is an integer (0/1) and `spend` is a float, both appropriate for the analysis.
- `date` is stored as text, so it will be converted to `datetime`.

A clean `info()` output is not enough to trust the data. The next checks look for duplicate users, unexpected categories and inconsistencies between `converted` and `spend`.

### 1.5 Data quality checks

**a) User IDs and completeness**

In [ ]:
# ============================================================
# 1.5a USER IDs AND COMPLETENESS
# ============================================================
total_rows = len(df)
unique_users = df["user_id"].nunique()
duplicate_ids = df["user_id"].duplicated().sum()
null_ids = df["user_id"].isna().sum()
duplicate_rows = df.duplicated().sum()
missing_values = df.isna().sum().sum()

print("🆔 USER IDs AND COMPLETENESS")
print("-" * 45)
print(f"Total rows:            {total_rows:,}")
print(f"Unique users:          {unique_users:,}")
print(f"Repeated user IDs:     {duplicate_ids:,}")
print(f"Null user IDs:         {null_ids:,}")
print(f"Fully duplicated rows: {duplicate_rows:,}")
print(f"Missing values (all):  {missing_values:,}")

assert duplicate_ids == 0 and null_ids == 0, "There are repeated or null user IDs"
assert missing_values == 0, "There are missing values in the dataset"
print("\n✅ Every row is a unique user and there are no missing values")

**b) Experiment dates**

In [ ]:
# ============================================================
# 1.5b EXPERIMENT DATES
# ============================================================
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

users_per_day = df.groupby("date").size()

print("📅 EXPERIMENT WINDOW")
print("-" * 45)
print(f"First date:     {df['date'].min():%Y-%m-%d}")
print(f"Last date:      {df['date'].max():%Y-%m-%d}")
print(f"Distinct dates: {df['date'].nunique()}")
print(f"Users per day:  min {users_per_day.min():,} | max {users_per_day.max():,} | "
      f"mean {users_per_day.mean():,.0f}")

**c) Categorical values**

In [ ]:
# ============================================================
# 1.5c CATEGORICAL VALUES
# ============================================================
EXPECTED_CATEGORIES = {
    "landing": {"A", "B"},
    "region": {"North", "Central", "South", "West", "East"},
    "device": {"Mobile", "Desktop"},
    "traffic_source": {"Organic", "Ads", "Email", "Referral"},
    "user_type": {"New", "Returning"},
}

for column, expected in EXPECTED_CATEGORIES.items():
    counts = df[column].value_counts()
    unexpected = set(counts.index) - expected
    print(f"📌 {column}")
    for category, n in counts.items():
        print(f"   {category:<10} {n:>7,}  ({n / len(df):.2%})")
    print("   ✅ Only expected categories\n" if not unexpected
          else f"   ⚠️ Unexpected categories: {sorted(unexpected)}\n")

assert set(df["landing"].unique()) == {"A", "B"}, "The landing column must only contain A and B"

**d) Consistency between `converted` and `spend`**

Business rule: **spend = 0 if and only if the user did not convert**. It is checked in both directions.

In [ ]:
# ============================================================
# 1.5d CONSISTENCY BETWEEN CONVERSION AND SPEND
# ============================================================
print("Unique values in converted:", sorted(int(v) for v in df["converted"].unique()))

not_converted_with_spend = ((df["converted"] == 0) & (df["spend"] > 0)).sum()
converted_without_spend = ((df["converted"] == 1) & (df["spend"] == 0)).sum()
negative_spend = (df["spend"] < 0).sum()

print(f"Non-buyers with spend > 0: {not_converted_with_spend:,}")
print(f"Buyers with spend = 0:     {converted_without_spend:,}")
print(f"Negative spend values:     {negative_spend:,}")

# Cross-tab: conversion vs. whether there was any spend
rule_table = pd.crosstab(df["converted"], df["spend"] > 0, margins=True, margins_name="Total")
rule_table = rule_table.rename(columns={False: "spend = 0", True: "spend > 0"})
display(rule_table)

assert set(df["converted"].unique()) <= {0, 1}, "converted must only contain 0 and 1"
assert not_converted_with_spend == 0 and converted_without_spend == 0, "The rule spend = 0 ⇔ not converted is broken"
assert negative_spend == 0, "There are negative spend values"
print("✅ The rule 'spend = 0 if and only if the user did not convert' holds for every row")

**e) Spend distribution and outliers**

In [ ]:
# ============================================================
# 1.5e SPEND DISTRIBUTION AND OUTLIERS
# ============================================================
spend_all = df["spend"]
buyer_spend = df.loc[df["converted"] == 1, "spend"]

print("💵 SPEND — ALL USERS")
print("-" * 45)
print(f"Mean:                  {spend_all.mean():.2f}")
print(f"Median:                {spend_all.median():.2f}")
print(f"Users with zero spend: {(spend_all == 0).sum():,} ({(spend_all == 0).mean():.2%})")

print("\n🛒 SPEND — BUYERS ONLY")
print("-" * 45)
print(f"Buyers:  {len(buyer_spend):,} ({len(buyer_spend) / len(df):.3%} of all users)")
print(f"Mean:    {buyer_spend.mean():.2f}")
print(f"Median:  {buyer_spend.median():.2f}")
print(f"Min:     {buyer_spend.min():.2f}")
print(f"Max:     {buyer_spend.max():.2f}")

# IQR rule
q1, q3 = buyer_spend.quantile([0.25, 0.75])
upper_limit = iqr_upper_limit(buyer_spend)
outliers = buyer_spend[buyer_spend > upper_limit]

print(f"\nQ1 = {q1:.2f} | Q3 = {q3:.2f} | IQR = {q3 - q1:.2f}")
print(f"Upper limit (Q3 + 1.5·IQR): {upper_limit:.2f}")
print(f"Outliers: {len(outliers):,} ({len(outliers) / len(buyer_spend):.2%} of buyers)")
print(f"Mean without outliers: {buyer_spend[buyer_spend <= upper_limit].mean():.2f}")

print("\nHigh percentiles (buyers):")
print(buyer_spend.quantile([0.50, 0.75, 0.90, 0.95, 0.99]).round(2).to_string())

# Charts
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].boxplot(buyer_spend, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=COLOR_NOT_CONVERTED),
                medianprops=dict(color="black"))
axes[0].axhline(upper_limit, color=COLOR_BELOW, linestyle="--",
                label=f"IQR limit ({upper_limit:.2f})")
axes[0].set_title("Spend among buyers (boxplot)")
axes[0].set_ylabel("Spend (monetary units)")
axes[0].set_xticks([1])
axes[0].set_xticklabels(["Buyers"])
axes[0].legend(frameon=False)

axes[1].hist(buyer_spend, bins=50, color=COLOR_NOT_CONVERTED, edgecolor="white")
axes[1].axvline(buyer_spend.mean(), color=COLOR_BELOW, linestyle="--",
                label=f"Mean ({buyer_spend.mean():.2f})")
axes[1].axvline(buyer_spend.median(), color=COLOR_B, linestyle="--",
                label=f"Median ({buyer_spend.median():.2f})")
axes[1].set_title("Spend among buyers (histogram)")
axes[1].set_xlabel("Spend (monetary units)")
axes[1].set_ylabel("Number of buyers")
axes[1].legend(frameon=False)

plt.tight_layout()
plt.show()

**📌 Reading the spend distribution:** most users never buy, so the median spend across all users is 0. Among buyers, spend is **right-skewed**: a long tail of large purchases sits above the IQR limit and pulls the mean above the median. For that reason spend is analyzed among buyers with Welch's t-test plus robustness checks (Step 2), and revenue per exposed user is analyzed with a bootstrap (Step 7).

### 1.6 Group balance: sample ratio and composition

In a well-randomized A/B test, both versions should receive a similar number of users (**sample ratio mismatch**, or SRM, check) and a similar mix of regions, devices, channels and user types.

In [ ]:
# ============================================================
# 1.6 GROUP BALANCE
# ============================================================
# 1) Sample ratio mismatch (SRM): are the group sizes compatible with a 50/50 split?
group_sizes = df["landing"].value_counts().sort_index()
srm_stat, srm_p = chisquare(group_sizes)

print("⚖️ GROUP SIZES")
print("-" * 45)
for version, n in group_sizes.items():
    print(f"Version {version}: {n:,} users ({n / group_sizes.sum():.3%})")
print(f"SRM chi-square: {srm_stat:.4f} | p-value: {srm_p:.4f}")
print("✅ No sample ratio mismatch\n" if srm_p >= ALPHA else "⚠️ Possible sample ratio mismatch\n")

# 2) Composition of each version by categorical variable
BALANCE_COLUMNS = ["region", "device", "traffic_source", "user_type"]
balance_rows = []

for column in BALANCE_COLUMNS:
    counts = pd.crosstab(df[column], df["landing"])
    shares = pd.crosstab(df[column], df["landing"], normalize="columns") * 100
    shares["Diff B − A (pp)"] = shares["B"] - shares["A"]
    chi2_bal, p_bal, _, _ = chi2_contingency(counts)

    print(f"📌 {column}: % of users within each version")
    display(shares.round(2))

    balance_rows.append({
        "Variable": column,
        "Max abs. diff (pp)": shares["Diff B − A (pp)"].abs().max(),
        "Chi-square": chi2_bal,
        "p-value": p_bal,
    })

# 3) Balance summary
balance_summary = pd.DataFrame(balance_rows).set_index("Variable")
balance_summary["Balanced?"] = np.where(balance_summary["p-value"] >= ALPHA, "✅ Yes", "⚠️ Check")
print("🧾 BALANCE SUMMARY")
display(balance_summary.round(4))

### 1.7 ✅ Exploratory summary

**1. Record integrity.** The dataset has **40,000 rows for 40,000 unique users**, with no repeated IDs, no duplicated rows and no missing values, so no user is counted twice.

**2. Experiment window.** Records run from **January 1 to January 28, 2026** (28 distinct dates). The `date` column was converted to `datetime`.

**3. Group assignment.** Version A received **19,982 users (49.955%)** and Version B **20,018 (50.045%)**. The SRM test shows no sample ratio mismatch (**p = 0.857**), and the composition by region, device, traffic source and user type is balanced between versions (**all p > 0.05**). The two groups are comparable.

**4. Spend.** Mean spend across all users is **9.33 monetary units**, while the median is 0: **85.74% of users did not buy**. The **5,706 buyers** (an overall conversion rate of **14.265%**) spent **65.37** on average, with a median of **59.86** and a range of **12.12 to 303.68**. The distribution is right-skewed, with outliers above the IQR limit of about **136.5**.

**5. Categories.** Only the expected labels appear.
- **Region:** North 11,166 · Central 9,613 · South 8,039 · West 6,398 · East 4,784.
- **Device:** Mobile 24,829 (62.07%) · Desktop 15,171 (37.93%).
- **Traffic source:** Organic 17,987 · Ads 11,935 · Email 6,123 · Referral 3,955.
- **User type:** New 26,033 (65.08%) · Returning 13,967 (34.92%).

**Conclusion:** the data is complete, consistent and well randomized. Because most users spend nothing and buyer spend has outliers, **conversion and spend are analyzed separately**, and spend is compared with methods that are robust to skew.

## 💰 Step 2: Average Spend per Buyer (A vs. B)

**Objective:** determine whether buyers spend a different amount on Version A and Version B.

Only **buyers** are compared here, which is justified by the rule validated in Step 1.5d: non-buyers always have spend = 0, so including them would mix two different questions (whether people buy and how much they spend).

**Test:** Welch's t-test for independent samples, which compares two means **without assuming equal variances**.

- **H₀:** mean spend per buyer is the same in both versions (μA = μB).
- **H₁:** mean spend per buyer differs between versions (μA ≠ μB).
- **Significance level:** α = 0.05, two-sided test.

The assumptions are checked **before** running the test.

### 2.1 Spend by version

In [ ]:
# ============================================================
# 2.1 SPEND PER BUYER BY VERSION
# ============================================================
buyers = df[df["converted"] == 1]
spend_A = buyers.loc[buyers["landing"] == "A", "spend"]
spend_B = buyers.loc[buyers["landing"] == "B", "spend"]

spend_summary = pd.DataFrame({
    "Version A": spend_A.describe(),
    "Version B": spend_B.describe(),
}).T.rename(columns={"count": "buyers"})

print("🛒 SPEND PER BUYER (monetary units)")
display(spend_summary.round(2))

### 2.2 Assumption checks for Welch's t-test

In [ ]:
# ============================================================
# 2.2 ASSUMPTION CHECKS
# ============================================================
# 1) Independence: each user appears once and in only one version
users_in_both = (df.groupby("user_id")["landing"].nunique() > 1).sum()
print("🔗 INDEPENDENCE")
print("-" * 60)
print(f"Unique buyers: {buyers['user_id'].nunique():,} of {len(buyers):,} rows")
print(f"Users assigned to both versions: {users_in_both}")
print("✅ Observations are independent\n" if users_in_both == 0
      else "⚠️ Some users appear in both versions\n")

# 2) Shape of the distribution
print("📐 SHAPE OF THE DISTRIBUTION")
print("-" * 60)
for name, values in [("Version A", spend_A), ("Version B", spend_B)]:
    print(f"{name}: n = {len(values):,} | skewness = {stats.skew(values):.2f} | "
          f"mean = {values.mean():.2f} | median = {values.median():.2f}")
print()

# 3) Equality of variances (Levene)
levene_stat, levene_p = stats.levene(spend_A, spend_B)
print("📊 VARIANCES (Levene's test)")
print("-" * 60)
print(f"Std. dev. A: {spend_A.std():.2f} | Std. dev. B: {spend_B.std():.2f}")
print(f"Statistic: {levene_stat:.4f} | p-value: {fmt_p(levene_p)}")
print("➡️ Variances differ: Welch's t-test is the right choice\n" if levene_p < ALPHA
      else "➡️ Variances are similar: Welch's t-test is still valid and safer\n")

# 4) Outliers per version (IQR rule)
spend_limits = {"A": iqr_upper_limit(spend_A), "B": iqr_upper_limit(spend_B)}
print("🚨 OUTLIERS (IQR rule)")
print("-" * 60)
for version, values in [("A", spend_A), ("B", spend_B)]:
    n_out = (values > spend_limits[version]).sum()
    print(f"Version {version}: limit = {spend_limits[version]:.2f} | outliers = {n_out} "
          f"({n_out / len(values):.2%}) | max = {values.max():.2f}")

# Charts: histograms, boxplots and Q-Q plots
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

bins = np.histogram_bin_edges(np.concatenate([spend_A, spend_B]), bins=50)
axes[0, 0].hist(spend_A, bins=bins, alpha=0.55, color=COLOR_A, label="Version A")
axes[0, 0].hist(spend_B, bins=bins, alpha=0.55, color=COLOR_B, label="Version B")
axes[0, 0].set_title("Spend per buyer by version")
axes[0, 0].set_xlabel("Spend (monetary units)")
axes[0, 0].set_ylabel("Number of buyers")
axes[0, 0].legend(frameon=False)

box = axes[0, 1].boxplot([spend_A, spend_B], widths=0.5, patch_artist=True,
                         medianprops=dict(color="black"))
for patch, color in zip(box["boxes"], [COLOR_A, COLOR_B]):
    patch.set_facecolor(color)
axes[0, 1].set_xticks([1, 2])
axes[0, 1].set_xticklabels(["Version A", "Version B"])
axes[0, 1].set_title("Spend per buyer (boxplot)")
axes[0, 1].set_ylabel("Spend (monetary units)")

for ax, values, color, name in [(axes[1, 0], spend_A, COLOR_A, "A"), (axes[1, 1], spend_B, COLOR_B, "B")]:
    stats.probplot(values, dist="norm", plot=ax)
    points, fit_line = ax.get_lines()[:2]
    points.set_markerfacecolor(color)
    points.set_markeredgecolor(color)
    points.set_markersize(3)
    fit_line.set_color("black")
    ax.set_title(f"Q-Q plot: Version {name}")
    ax.set_xlabel("Theoretical quantiles (normal)")
    ax.set_ylabel("Observed spend")

plt.tight_layout()
plt.show()

**📌 Assumption check summary**

- **Independence:** every user appears once and in only one version. ✅
- **Shape:** spend is right-skewed in both versions (skewness of 1.67 in A and 1.20 in B), and the Q-Q plots show the upper tail departing from the normal line.
- **Variances:** Levene's test rejects equal variances (p ≈ 7e-08), so **Welch's t-test**, which does not assume equal variances, is the right choice.
- **Outliers:** about 2.8% of buyers in each version are above the IQR limit.
- **Sample size:** with **2,512 and 3,194 buyers**, the Central Limit Theorem makes the sample means approximately normal even though individual spend is skewed, so Welch's test is reliable. Robustness checks in Step 2.4 confirm this.

### 2.3 Welch's t-test

In [ ]:
# ============================================================
# 2.3 WELCH'S T-TEST: SPEND PER BUYER
# ============================================================
# Version B is passed first, so a positive t means B spends more than A.
# Every difference in this notebook is reported as B − A.
t_spend, p_spend = ttest_ind(spend_B, spend_A, equal_var=False)

mean_spend_A = spend_A.mean()
mean_spend_B = spend_B.mean()
diff_spend = mean_spend_B - mean_spend_A
lift_spend = diff_spend / mean_spend_A * 100

# 95% confidence interval for B − A (Welch–Satterthwaite)
var_mean_A = spend_A.var(ddof=1) / len(spend_A)
var_mean_B = spend_B.var(ddof=1) / len(spend_B)
se_spend = np.sqrt(var_mean_A + var_mean_B)
dof_spend = (var_mean_A + var_mean_B) ** 2 / (
    var_mean_A ** 2 / (len(spend_A) - 1) + var_mean_B ** 2 / (len(spend_B) - 1)
)
ci_spend_low, ci_spend_high = student_t.interval(0.95, dof_spend, loc=diff_spend, scale=se_spend)

print("💰 WELCH'S T-TEST: SPEND PER BUYER")
print("-" * 55)
print(f"Buyers A: {len(spend_A):,} | Buyers B: {len(spend_B):,}")
print(f"Mean spend A: {mean_spend_A:.2f}")
print(f"Mean spend B: {mean_spend_B:.2f}")
print(f"Difference (B − A): {diff_spend:+.2f} monetary units ({lift_spend:+.1f}%)")
print(f"95% CI for B − A: [{ci_spend_low:+.2f}, {ci_spend_high:+.2f}]")
print(f"t statistic: {t_spend:.4f} | degrees of freedom: {dof_spend:,.1f}")
print(f"p-value: {fmt_p(p_spend)}")
print(decision(p_spend))

### 2.4 Robustness checks

Because spend is skewed and has outliers, three alternative methods are used to confirm the Welch result:

1. **Bootstrap 95% CI** of the difference in means, which makes no distributional assumption.
2. **Mann-Whitney U test**, which compares distributions using ranks instead of means.
3. **Welch's t-test without outliers**, to make sure a few large purchases are not driving the result.

In [ ]:
# ============================================================
# 2.4 ROBUSTNESS CHECKS
# ============================================================
# 1) Bootstrap 95% CI of the difference in means (B − A)
rng_spend = np.random.default_rng(SEED)
values_A = spend_A.to_numpy()
values_B = spend_B.to_numpy()

boot_spend_diffs = np.empty(N_BOOT)
for i in range(N_BOOT):
    boot_spend_diffs[i] = (rng_spend.choice(values_B, len(values_B)).mean()
                           - rng_spend.choice(values_A, len(values_A)).mean())
boot_spend_low, boot_spend_high = np.percentile(boot_spend_diffs, [2.5, 97.5])

# 2) Mann-Whitney U test
u_stat, p_mw = mannwhitneyu(spend_B, spend_A, alternative="two-sided")

# 3) Welch's t-test without outliers (IQR limit of each version)
spend_A_trim = spend_A[spend_A <= spend_limits["A"]]
spend_B_trim = spend_B[spend_B <= spend_limits["B"]]
t_trim, p_trim = ttest_ind(spend_B_trim, spend_A_trim, equal_var=False)

robustness = pd.DataFrame({
    "Method": [
        "Welch (main test)",
        "Bootstrap 95% CI (B − A)",
        "Mann-Whitney U",
        "Welch without outliers",
    ],
    "Result": [
        f"Diff = {diff_spend:+.2f}",
        f"[{boot_spend_low:+.2f}, {boot_spend_high:+.2f}]",
        f"Median A = {spend_A.median():.2f} | Median B = {spend_B.median():.2f}",
        f"Diff = {spend_B_trim.mean() - spend_A_trim.mean():+.2f}",
    ],
    "p-value": [fmt_p(p_spend), "n/a", fmt_p(p_mw), fmt_p(p_trim)],
    "Significant?": [
        "✅ Yes" if p_spend < ALPHA else "❌ No",
        "✅ Yes" if boot_spend_low > 0 or boot_spend_high < 0 else "❌ No",
        "✅ Yes" if p_mw < ALPHA else "❌ No",
        "✅ Yes" if p_trim < ALPHA else "❌ No",
    ],
}).set_index("Method")

print("🛡️ ROBUSTNESS CHECKS")
display(robustness)

### 2.5 📝 Conclusion: spend per buyer

| Metric | Version A (control) | Version B (variant) |
|---|---:|---:|
| Buyers | 2,512 | 3,194 |
| Mean spend per buyer (monetary units) | 61.09 | 68.75 |

- **Welch's t statistic:** 9.4810 (positive because Version B is passed first, meaning B > A)
- **p-value:** < 0.001
- **Difference (B − A):** **+7.66 monetary units**, 95% CI **[+6.08, +9.24]**
- **Relative increase:** **+12.5%** versus Version A

**✅ Statistical decision:** the p-value is far below 0.05, so **H₀ is rejected**. The confidence interval does not include 0, which supports a real difference in mean spend per buyer. This holds under the assumptions checked in Step 2.2: observations are independent, spend is right-skewed with about 2.8% outliers per version, and both groups are large (more than 2,500 buyers each), so Welch's test is reasonable. The bootstrap, Mann-Whitney and Welch-without-outliers checks all agree with the main test.

**💼 Business reading:** buyers on Version B spend **7.66 monetary units more** on average (+12.5%). This only covers people who bought, though. A higher spend per buyer does not by itself mean more revenue per visitor; the next step checks whether B also converts more visitors.

## 🎯 Step 3: Conversion Rate (A vs. B)

**Objective:** determine whether the share of visitors who buy differs between Version A and Version B.

**Test:** two-proportion z-test.

- **H₀:** the conversion rates are equal (pA = pB).
- **H₁:** the conversion rates differ (pA ≠ pB).
- **Significance level:** α = 0.05, two-sided test.

### 3.1 Conversions by version

In [ ]:
# ============================================================
# 3.1 CONVERSIONS BY VERSION
# ============================================================
conversion_summary = df.groupby("landing")["converted"].agg(users="size", conversions="sum")
conversion_summary["non_conversions"] = conversion_summary["users"] - conversion_summary["conversions"]
conversion_summary["conversion_rate_%"] = conversion_summary["conversions"] / conversion_summary["users"] * 100

conv_A, conv_B = int(conversion_summary.loc["A", "conversions"]), int(conversion_summary.loc["B", "conversions"])
n_A, n_B = int(conversion_summary.loc["A", "users"]), int(conversion_summary.loc["B", "users"])
rate_A, rate_B = conv_A / n_A, conv_B / n_B
diff_rate = rate_B - rate_A
lift_rate = diff_rate / rate_A * 100

print("🎯 CONVERSION BY VERSION")
display(conversion_summary.round(2))
print(f"Conversion rate A: {rate_A:.2%}")
print(f"Conversion rate B: {rate_B:.2%}")
print(f"Difference (B − A): {diff_rate * 100:+.2f} percentage points")
print(f"Relative change B vs. A: {lift_rate:+.2f}%")

### 3.2 Assumption checks for the z-test

In [ ]:
# ============================================================
# 3.2 ASSUMPTION CHECKS
# ============================================================
print("🔍 ASSUMPTION CHECKS")
print("-" * 60)

# a) Normal approximation: at least 10 successes and 10 failures per group
normal_ok = True
for version, conversions, users in [("A", conv_A, n_A), ("B", conv_B, n_B)]:
    failures = users - conversions
    ok = conversions >= 10 and failures >= 10
    normal_ok = normal_ok and ok
    print(f"Version {version}: n = {users:,} | conversions = {conversions:,} | "
          f"non-conversions = {failures:,} {'✅' if ok else '⚠️'}")
print("✅ Normal approximation holds (≥ 10 successes and ≥ 10 failures per group)"
      if normal_ok else "⚠️ Normal approximation does not hold")

# b) Independence: each user appears once and in only one version
repeated_users = df["user_id"].duplicated().sum()
users_in_both = (df.groupby("user_id")["landing"].nunique() > 1).sum()
independent = repeated_users == 0 and users_in_both == 0
print(f"\nRepeated users: {repeated_users} | Users in both versions: {users_in_both}")
print("✅ Observations are independent" if independent else "⚠️ Observations are not independent")

if not (normal_ok and independent):
    raise ValueError("The z-test assumptions are not met.")

### 3.3 Two-proportion z-test and confidence interval

In [ ]:
# ============================================================
# 3.3 TWO-PROPORTION Z-TEST
# ============================================================
# Version B is passed first, so a positive z means B converts more than A.
z_stat, p_conv = proportions_ztest(count=[conv_B, conv_A], nobs=[n_B, n_A], alternative="two-sided")

# 95% Wald confidence interval for the difference B − A
se_rate = np.sqrt(rate_A * (1 - rate_A) / n_A + rate_B * (1 - rate_B) / n_B)
z_crit = norm.ppf(1 - ALPHA / 2)
ci_rate_low = diff_rate - z_crit * se_rate
ci_rate_high = diff_rate + z_crit * se_rate

print("📊 TWO-PROPORTION Z-TEST")
print("-" * 55)
print(f"Conversion rate A: {rate_A:.2%}")
print(f"Conversion rate B: {rate_B:.2%}")
print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {fmt_p(p_conv)}")
print(decision(p_conv))

print(f"\n📏 Difference (B − A): {diff_rate * 100:+.2f} percentage points")
print(f"95% CI: [{ci_rate_low * 100:+.2f}, {ci_rate_high * 100:+.2f}] percentage points")
print("➡️ The interval does not include 0" if ci_rate_low > 0 or ci_rate_high < 0
      else "➡️ The interval includes 0")
print(f"\n🧮 Extra conversions per 1,000 visitors with B: {diff_rate * 1000:.1f}")

### 3.4 Consistency over time

A result that only comes from a few unusual days would be fragile. The chart compares the daily conversion rate of each version across the 28 days of the experiment.

In [ ]:
# ============================================================
# 3.4 DAILY CONVERSION RATE BY VERSION
# ============================================================
daily_rates = df.groupby(["date", "landing"])["converted"].mean().unstack("landing") * 100
days_B_ahead = int((daily_rates["B"] > daily_rates["A"]).sum())
total_days = len(daily_rates)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_rates.index, daily_rates["A"], marker="o", markersize=4, color=COLOR_A, label="Version A")
ax.plot(daily_rates.index, daily_rates["B"], marker="o", markersize=4, color=COLOR_B, label="Version B")
ax.axhline(rate_A * 100, color=COLOR_A, linestyle="--", linewidth=1, label=f"A overall ({rate_A:.2%})")
ax.axhline(rate_B * 100, color=COLOR_B, linestyle="--", linewidth=1, label=f"B overall ({rate_B:.2%})")
ax.set_title("Daily conversion rate by version")
ax.set_xlabel("Date")
ax.set_ylabel("Conversion rate (%)")
ax.legend(frameon=False, ncol=2)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"📅 Version B had a higher conversion rate than A on {days_B_ahead} of {total_days} days "
      f"({days_B_ahead / total_days:.0%}).")

**📌 Reading the daily chart:** with roughly 700 users per version per day, daily rates are noisy, so single days can cross. What matters is the overall pattern: the printed count shows on how many days B beat A. When B stays ahead on most days, its advantage is not driven by a handful of unusual days. The experiment still only lasted 28 days, so longer-term effects (such as a novelty effect wearing off) cannot be ruled out; this is listed as a limitation in Step 8.

### 3.5 📝 Conclusion: conversion rate

| Metric | Version A (control) | Version B (variant) |
|---|---:|---:|
| Users exposed | 19,982 | 20,018 |
| Conversions | 2,512 | 3,194 |
| Conversion rate | 12.57% | 15.96% |

- **z statistic:** 9.6774 (positive because Version B is passed first, meaning B > A)
- **p-value:** 3.76e-22
- **Difference (B − A):** **+3.38 percentage points**, 95% CI **[+2.70, +4.07]** pp
- **Relative increase:** **+26.92%** versus Version A

**✅ Statistical decision:** the p-value is far below 0.05, so **H₀ is rejected**. Version B has a significantly higher conversion rate, and the confidence interval is entirely above 0.

**💼 Business reading:** for every **1,000 visitors**, Version B produces about **34 more buyers** than Version A. Combined with Step 2 (buyers on B also spend more), B looks stronger on both levers. Step 7 combines them into a single metric: revenue per exposed user.

## 🌐 Step 4: Traffic Source vs. Conversion

**Objective:** check whether conversion depends on the acquisition channel (`traffic_source`).

**Test:** chi-square test of independence, plus **Cramér's V** to measure the size of any association.

- **H₀:** traffic source and conversion are independent; the conversion rate is the same across channels.
- **H₁:** traffic source and conversion are associated; at least one channel converts differently.
- **Significance level:** α = 0.05.

An association here would **not prove causation**: channels differ in audience, intent and cost.

### 4.1 Contingency table and channel performance

In [ ]:
# ============================================================
# 4.1 CONTINGENCY TABLE: TRAFFIC SOURCE × CONVERSION
# ============================================================
traffic_table = pd.crosstab(df["traffic_source"], df["converted"])
traffic_order = traffic_table.sum(axis=1).sort_values(ascending=False).index.tolist()
traffic_table = traffic_table.loc[traffic_order, [0, 1]]   # reused in Step 6

overall_rate = df["converted"].mean() * 100

traffic_summary = pd.DataFrame({
    "Users": traffic_table.sum(axis=1),
    "Conversions": traffic_table[1],
})
traffic_summary["Conversion rate (%)"] = traffic_summary["Conversions"] / traffic_summary["Users"] * 100
traffic_summary["vs. overall (pp)"] = traffic_summary["Conversion rate (%)"] - overall_rate
traffic_summary["Share of conversions (%)"] = traffic_summary["Conversions"] / traffic_summary["Conversions"].sum() * 100
traffic_summary = traffic_summary.sort_values("Conversion rate (%)", ascending=False)

print("🌐 PERFORMANCE BY TRAFFIC SOURCE")
display(traffic_summary.round(2))

best_rate_channel = traffic_summary["Conversion rate (%)"].idxmax()
worst_rate_channel = traffic_summary["Conversion rate (%)"].idxmin()
top_volume_channel = traffic_summary["Conversions"].idxmax()
channel_gap = traffic_summary["Conversion rate (%)"].max() - traffic_summary["Conversion rate (%)"].min()

print(f"🌍 Overall conversion rate: {overall_rate:.2f}%")
print(f"🏅 Highest conversion rate: {best_rate_channel}")
print(f"📦 Most conversions: {top_volume_channel} "
      f"({traffic_summary.loc[top_volume_channel, 'Share of conversions (%)']:.2f}% of all conversions)")
print(f"↔️ Gap between {best_rate_channel} and {worst_rate_channel}: {channel_gap:.2f} percentage points")

### 4.2 Chi-square test and effect size

In [ ]:
# ============================================================
# 4.2 CHI-SQUARE TEST + CRAMÉR'S V
# ============================================================
chi2_traffic, p_traffic, dof_traffic, expected_traffic = chi2_contingency(traffic_table)

# Cramér's V = sqrt( χ² / (n · (min(rows, columns) − 1)) )
n_traffic = traffic_table.to_numpy().sum()
v_traffic = np.sqrt(chi2_traffic / (n_traffic * (min(traffic_table.shape) - 1)))

print("🧪 CHI-SQUARE TEST: TRAFFIC SOURCE × CONVERSION")
print("-" * 55)
print(f"Chi-square statistic: {chi2_traffic:.4f}")
print(f"p-value: {fmt_p(p_traffic)}")
print(f"Degrees of freedom: {dof_traffic}")
print(decision(p_traffic))

print("\n🔍 ASSUMPTION CHECK")
print(f"Minimum expected frequency: {expected_traffic.min():.2f}")
print(f"Cells with expected frequency < 5: {(expected_traffic < 5).sum()}")
print("✅ All expected frequencies are at least 5" if (expected_traffic >= 5).all()
      else "⚠️ Some expected frequencies are below 5")

print("\n📏 EFFECT SIZE")
print(f"Cramér's V: {v_traffic:.4f} → association is {cramers_v_label(v_traffic)}")

### 4.3 📝 Conclusion: traffic source

- **Chi-square:** 8.6621 · **p-value:** 0.0341 · **Degrees of freedom:** 3
- **Minimum expected frequency:** 564.18 (all cells above 5, so the test is valid)
- **Cramér's V:** 0.0147 (computed in Step 4.2)

**✅ Statistical decision:** p = 0.0341 < 0.05, so **H₀ is rejected**: there is a statistically significant association between traffic source and conversion. However, the **evidence is moderate** (the p-value is close to 0.05) and **the effect is negligible** (Cramér's V = 0.0147). The result is detectable mainly because the sample is large.

**💼 Business reading**

| Channel | Conversions | Conversion rate |
|---|---:|---:|
| Email | 918 | 14.99% |
| Ads | 1,759 | 14.74% |
| Referral | 549 | 13.88% |
| Organic | 2,480 | 13.79% |

- **Email** has the highest observed conversion rate, while **Organic** brings the most buyers (**43.46%** of all conversions).
- Only **1.20 percentage points** separate the best channel (Email) from the worst (Organic).
- Volume and efficiency do not go together: the largest channel is not the most efficient one.

**🎯 Conclusion:** the global chi-square test does not show that Email beats every other channel individually; that would require pairwise comparisons with a multiple-testing correction. These findings are a starting point for investigation, not a reason to move budget, which would also require acquisition cost and profitability data per channel.

## 👤 Step 5: User Type vs. Conversion

**Objective:** check whether new and returning users convert at different rates. A returning user may have visited before without having bought.

**Test:** chi-square test of independence, plus Cramér's V.

- **H₀:** user type and conversion are independent; new and returning users convert at the same rate.
- **H₁:** user type and conversion are associated.
- **Significance level:** α = 0.05.

### 5.1 Contingency table and conversion rates

In [ ]:
# ============================================================
# 5.1 CONTINGENCY TABLE: USER TYPE × CONVERSION
# ============================================================
user_table = pd.crosstab(df["user_type"], df["converted"])
user_order = user_table.sum(axis=1).sort_values(ascending=False).index.tolist()
user_table = user_table.loc[user_order, [0, 1]]   # reused in Step 6

user_summary = pd.DataFrame({
    "Users": user_table.sum(axis=1),
    "Not converted": user_table[0],
    "Converted": user_table[1],
})
user_summary["Conversion rate (%)"] = user_summary["Converted"] / user_summary["Users"] * 100
user_summary["Share of conversions (%)"] = user_summary["Converted"] / user_summary["Converted"].sum() * 100

print("👤 PERFORMANCE BY USER TYPE")
display(user_summary.round(2))

segment_gap = user_summary["Conversion rate (%)"].max() - user_summary["Conversion rate (%)"].min()
print(f"🏅 Highest conversion rate: {user_summary['Conversion rate (%)'].idxmax()}")
print(f"↔️ Gap between segments: {segment_gap:.2f} percentage points")

# Descriptive view: conversion rate by segment within each version
rates_by_page = pd.crosstab(df["user_type"], df["landing"], values=df["converted"], aggfunc="mean") * 100
rates_by_page = rates_by_page.loc[user_order]
rates_by_page["Diff B − A (pp)"] = rates_by_page["B"] - rates_by_page["A"]
print("\n📄 CONVERSION RATE (%) BY USER TYPE AND VERSION")
display(rates_by_page.round(2))

### 5.2 Chi-square test and effect size

In [ ]:
# ============================================================
# 5.2 CHI-SQUARE TEST + CRAMÉR'S V
# ============================================================
chi2_user, p_user, dof_user, expected_user = chi2_contingency(user_table)

n_user = user_table.to_numpy().sum()
v_user = np.sqrt(chi2_user / (n_user * (min(user_table.shape) - 1)))

print("🧪 CHI-SQUARE TEST: USER TYPE × CONVERSION")
print("-" * 55)
print(f"Chi-square statistic: {chi2_user:.4f}")
print(f"p-value: {fmt_p(p_user)}")
print(f"Degrees of freedom: {dof_user}")
print(decision(p_user))

print("\n🔍 ASSUMPTION CHECK")
print(f"Minimum expected frequency: {expected_user.min():,.2f}")
print("✅ All expected frequencies are at least 5" if (expected_user >= 5).all()
      else "⚠️ Some expected frequencies are below 5")

print("\n📏 EFFECT SIZE")
print(f"Cramér's V: {v_user:.4f} → association is {cramers_v_label(v_user)}")

### 5.3 📝 Conclusion: user type

- **Chi-square:** 0.5135 · **p-value:** 0.4736 · **Degrees of freedom:** 1
- **Cramér's V:** 0.0036 · **Minimum expected frequency:** 1,992.39

**✅ Statistical decision:** p = 0.4736 > 0.05, so **H₀ is not rejected**. There is no evidence that conversion depends on user type, and Cramér's V is practically zero.

**💼 Business reading**

- **New users:** 3,738 conversions out of 26,033 (**14.36%**).
- **Returning users:** 1,968 conversions out of 13,967 (**14.09%**).
- The gap is only **0.27 percentage points**. New users bring more buyers in absolute terms simply because they are the larger segment.

Looking at each version separately:

| Segment | Version A | Version B |
|---|---:|---:|
| New | 12.73% | 15.98% |
| Returning | 12.27% | 15.91% |

Version B converts better in **both** segments. This is descriptive: showing that B's lift is the same for both segments would require a formal interaction test.

**🎯 Conclusion:** the data does not support building separate conversion strategies for new and returning users based on this classification alone.

## 📊 Step 6: Visualizing the Categorical Results

The charts below show, for traffic source and user type:
- **absolute counts** of users who did and did not convert (volume), and
- **proportions** within each category (efficiency).

They reuse the contingency tables built in Steps 4 and 5 (`traffic_table` and `user_table`), so nothing is recalculated.

In [ ]:
# ============================================================
# 6.0 CHART HELPERS (used for both traffic source and user type)
# ============================================================
def grouped_count_chart(table, title, xlabel):
    """Side-by-side bars: users who did not convert vs. users who converted."""
    categories = table.index.tolist()
    x = np.arange(len(categories))
    width = 0.38

    fig, ax = plt.subplots(figsize=(11, 6))
    bars_no = ax.bar(x - width / 2, table[0], width, color=COLOR_NOT_CONVERTED, label="Did not convert")
    bars_yes = ax.bar(x + width / 2, table[1], width, color=COLOR_CONVERTED, label="Converted")

    for bar in list(bars_no) + list(bars_yes):
        ax.annotate(f"{int(bar.get_height()):,}",
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of users")
    ax.legend(title="Outcome", frameon=False)
    ax.margins(y=0.12)
    plt.tight_layout()
    plt.show()


def stacked_share_chart(table, title, xlabel):
    """100% stacked bars: share of users who did not convert vs. converted."""
    shares = table.div(table.sum(axis=1), axis=0) * 100
    categories = shares.index.tolist()
    x = np.arange(len(categories))

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(x, shares[0], width=0.6, color=COLOR_NOT_CONVERTED, label="Did not convert")
    ax.bar(x, shares[1], width=0.6, bottom=shares[0], color=COLOR_CONVERTED, label="Converted")

    for i, category in enumerate(categories):
        ax.text(i, shares.loc[category, 0] + shares.loc[category, 1] / 2,
                f"{shares.loc[category, 1]:.2f}%",
                ha="center", va="center", color="white", fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 100)
    ax.set_yticks(range(0, 101, 20))
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Share of users (%)")
    ax.legend(title="Outcome", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    plt.show()


print("✅ Chart helpers ready")

### 6.1 Traffic source and conversion

In [ ]:
# ============================================================
# 6.1a USERS BY TRAFFIC SOURCE AND OUTCOME (counts)
# ============================================================
print(f"📌 Channels ordered by volume: {' > '.join(traffic_order)}")
grouped_count_chart(traffic_table, "Users by traffic source and outcome", "Traffic source")

**📌 Reading the count chart:** **Organic brings the most users (17,987)**, followed by Ads (11,935), Email (6,123) and Referral (3,955). Larger channels naturally produce more conversions in absolute terms even when their conversion rate is not the highest, so the number of conversions must be read relative to each channel's volume. The next chart does exactly that.

In [ ]:
# ============================================================
# 6.1b CONVERSION SHARE BY TRAFFIC SOURCE (100% stacked)
# ============================================================
stacked_share_chart(traffic_table, "Conversion share by traffic source", "Traffic source")

**📌 Reading the stacked chart:** each bar represents **100% of the users of its channel**, so differences in volume no longer distort the comparison. The converted share ranges from **13.79% (Organic) to 14.99% (Email)**. On a 0–100% axis that gap of about 1.2 points is almost invisible, so the zoomed view below focuses on the conversion rate alone.

### 🔍 Zoomed view: conversion rate by channel

This chart shows only the conversion rate of each channel, with a reference line for the overall rate (14.27%) and error bars showing the 95% confidence interval.

**Note:** the vertical axis **starts at 12%, not 0%**, so that differences of about one percentage point are visible. This visually magnifies the differences, so bar heights should not be read as proportional; what matters is each value and its distance to the overall line.

In [ ]:
# ============================================================
# 6.1c CONVERSION RATE BY CHANNEL VS. OVERALL RATE (zoomed)
# ============================================================
channel_users = traffic_table.sum(axis=1)
channel_rates = traffic_table[1] / channel_users * 100
channel_errors = norm.ppf(0.975) * np.sqrt(
    (channel_rates / 100) * (1 - channel_rates / 100) / channel_users) * 100

colors = [COLOR_ABOVE if rate > overall_rate else COLOR_BELOW for rate in channel_rates]
y_low = min(12, np.floor((channel_rates - channel_errors).min()))
y_high = max(16.5, (channel_rates + channel_errors).max() + 0.8)

fig, ax = plt.subplots(figsize=(10, 6.5))
bars = ax.bar(channel_rates.index, channel_rates.values, yerr=channel_errors.values, capsize=8,
              color=colors, alpha=0.85, edgecolor="black", linewidth=0.6)
ax.axhline(overall_rate, color="black", linestyle="--", linewidth=1.5,
           label=f"Overall rate ({overall_rate:.2f}%)")

# Labels placed above the error bars so they never overlap
for bar, rate, err in zip(bars, channel_rates, channel_errors):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + err + 0.12,
            f"{rate:.2f}%", ha="center", fontweight="bold")

ax.set_ylim(y_low, y_high)
ax.set_title("Conversion rate by traffic source vs. overall rate")
ax.set_xlabel("Traffic source")
ax.set_ylabel(f"Conversion rate (%) — axis starts at {y_low:g}%")
ax.legend(loc="upper right", frameon=False)

# Footnote with reserved space so it does not overlap the axis
plt.tight_layout()
fig.subplots_adjust(bottom=0.2)
fig.text(0.5, 0.03,
         "Teal: above the overall rate | Orange: below | Error bars: 95% CI | "
         f"Vertical axis starts at {y_low:g}%",
         ha="center", fontsize=10, color="dimgray")
plt.show()

print("📏 Distance to the overall rate (percentage points):")
print((channel_rates - overall_rate).sort_values(ascending=False).round(2).to_string())

### 📈 Interpretation: conversion efficiency by channel

| Channel | Conversion rate | vs. overall rate (14.27%) |
|---|---:|---:|
| Email | 14.99% | +0.73 pp |
| Ads | 14.74% | +0.47 pp |
| Referral | 13.88% | −0.38 pp |
| Organic | 13.79% | −0.48 pp |

- Email and Ads convert above the overall rate; Referral and Organic fall below it.
- However, **the 95% confidence intervals of all four channels touch or cross the overall-rate line**, so no single channel clearly stands apart from the average on its own.
- The gap between the best channel (Email) and the worst (Organic) is **only 1.20 percentage points**. It is almost invisible in the stacked chart, and the zoomed chart makes it readable at the cost of magnifying it visually.

**🔗 Link to the chi-square test (Step 4):** χ² = 8.6621, p = 0.0341, Cramér's V = 0.0147. The difference between channels is **statistically detectable but practically small**, and the evidence is moderate.

**💼 Business implications:** Email may be worth investigating as an efficient channel, but a higher conversion rate does not automatically mean higher profitability, and **an association between channel and conversion does not prove causation**.

### 6.2 User type and conversion

In [ ]:
# ============================================================
# 6.2a USERS BY TYPE AND OUTCOME (counts)
# ============================================================
print(f"📌 Segments ordered by volume: {' > '.join(user_order)}")
grouped_count_chart(user_table, "Users by type and outcome", "User type")

**📌 Reading the count chart:** **New** is by far the largest segment (26,033 users vs. 13,967 Returning), so it contributes more conversions in absolute terms (3,738 vs. 1,968). This **does not mean new users convert better**: the conversion rates are 14.36% and 14.09%, as the next chart shows.

In [ ]:
# ============================================================
# 6.2b CONVERSION SHARE BY USER TYPE (100% stacked)
# ============================================================
stacked_share_chart(user_table, "Conversion share by user type", "User type")
print(f"↔️ Gap between segments: {segment_gap:.2f} percentage points")

### 📈 Interpretation: conversion efficiency by user type

Once each segment is scaled to 100%, the two bars look almost identical: **14.36% for New** and **14.09% for Returning**, a gap of only **0.27 percentage points**. This matches the chi-square test in Step 5 (p = 0.4736, Cramér's V = 0.0036): there is no evidence that user type is associated with conversion.

From a business perspective, the new/returning split is **not a useful lever for conversion** in this experiment. Any remaining differences could also reflect other characteristics, such as channel, region or device, so they should not be read as causal.

## 💵 Step 7: Revenue per Exposed User (A vs. B)

Revenue per exposed user (ARPU, total spend divided by all users shown the page) **combines conversion and spend** in a single number, which makes it the most relevant metric for the business decision. The goal is to evaluate whether the difference between versions is compatible with chance, using methods suited to its distribution.

**Shape of the distribution:** most users have revenue 0 (they did not buy), and among those who buy there are large values. The distribution is therefore extremely right-skewed.

**Methods:**
1. **Bootstrap (main method):** resamples the data 10,000 times to build 95% confidence intervals for the absolute difference and the relative change (%). It makes no distributional assumption, so it handles many zeros and heavy skew.
2. **Welch's t-test (complementary):** with about 20,000 users per group, the Central Limit Theorem makes the sample mean approximately normal even though individual values are not.

Mann-Whitney is not used here: with most values tied at zero, it compares ranks rather than average revenue, which is the metric of interest.

In [ ]:
# ============================================================
# 7.1 REVENUE PER EXPOSED USER: BOOTSTRAP + WELCH
# ============================================================
revenue_A = df.loc[df["landing"] == "A", "spend"].to_numpy()
revenue_B = df.loc[df["landing"] == "B", "spend"].to_numpy()

arpu_A = revenue_A.mean()
arpu_B = revenue_B.mean()
arpu_diff = arpu_B - arpu_A
arpu_lift = (arpu_B / arpu_A - 1) * 100

# 1) Shape of the distribution
print("📐 SHAPE OF REVENUE PER USER")
print("-" * 70)
for name, values in [("Version A", revenue_A), ("Version B", revenue_B)]:
    print(f"{name}: n = {len(values):,} | zero revenue = {(values == 0).mean():.2%} | "
          f"mean = {values.mean():.4f} | median = {np.median(values):.2f} | "
          f"skewness = {stats.skew(values):.2f}")

# 2) Bootstrap (main method)
rng_arpu = np.random.default_rng(SEED)
boot_arpu_diffs = np.empty(N_BOOT)
boot_arpu_lifts = np.empty(N_BOOT)

for i in range(N_BOOT):
    boot_mean_A = rng_arpu.choice(revenue_A, len(revenue_A)).mean()
    boot_mean_B = rng_arpu.choice(revenue_B, len(revenue_B)).mean()
    boot_arpu_diffs[i] = boot_mean_B - boot_mean_A
    boot_arpu_lifts[i] = (boot_mean_B / boot_mean_A - 1) * 100

ci_arpu_diff = np.percentile(boot_arpu_diffs, [2.5, 97.5])
ci_arpu_lift = np.percentile(boot_arpu_lifts, [2.5, 97.5])

# Two-sided bootstrap p-value: share of resamples on the other side of 0
p_boot = 2 * min((boot_arpu_diffs <= 0).mean(), (boot_arpu_diffs >= 0).mean())
p_boot_text = f"< {1 / N_BOOT:.0e}" if p_boot == 0 else f"{p_boot:.4f}"

print(f"\n🔁 BOOTSTRAP ({N_BOOT:,} resamples)")
print("-" * 70)
print(f"Revenue per user A: {arpu_A:.4f}")
print(f"Revenue per user B: {arpu_B:.4f}")
print(f"Difference (B − A): {arpu_diff:+.4f} monetary units per user")
print(f"95% CI of the difference: [{ci_arpu_diff[0]:+.4f}, {ci_arpu_diff[1]:+.4f}]")
print(f"Relative change B vs. A: {arpu_lift:+.2f}%")
print(f"95% CI of the relative change: [{ci_arpu_lift[0]:+.2f}%, {ci_arpu_lift[1]:+.2f}%]")
print(f"Bootstrap p-value: {p_boot_text}")

# 3) Welch's t-test (complementary). B is passed first, so a positive t means B > A.
t_arpu, p_arpu = ttest_ind(revenue_B, revenue_A, equal_var=False)
print("\n📊 WELCH'S T-TEST (all exposed users)")
print("-" * 70)
print(f"t statistic: {t_arpu:.4f}")
print(f"p-value: {fmt_p(p_arpu)}")

# 4) Where does the lift come from? ARPU = conversion rate × spend per buyer
conversion_factor = rate_B / rate_A
spend_factor = mean_spend_B / mean_spend_A
print("\n🧩 DECOMPOSITION OF THE LIFT")
print("-" * 70)
print(f"Conversion rate factor (B / A):  × {conversion_factor:.4f}")
print(f"Spend per buyer factor (B / A):  × {spend_factor:.4f}")
print(f"Revenue per user factor:         × {conversion_factor * spend_factor:.4f}  "
      f"(= {(conversion_factor * spend_factor - 1) * 100:+.2f}%)")

# 5) Summary
bootstrap_significant = ci_arpu_diff[0] > 0 or ci_arpu_diff[1] < 0
arpu_results = pd.DataFrame({
    "Method": ["Bootstrap (95% CI)", "Welch's t-test"],
    "Result": [
        f"[{ci_arpu_diff[0]:+.2f}, {ci_arpu_diff[1]:+.2f}] | [{ci_arpu_lift[0]:+.1f}%, {ci_arpu_lift[1]:+.1f}%]",
        f"t = {t_arpu:.2f}",
    ],
    "p-value": [p_boot_text, fmt_p(p_arpu)],
    "Significant?": [
        "✅ Yes" if bootstrap_significant else "❌ No",
        "✅ Yes" if p_arpu < ALPHA else "❌ No",
    ],
}).set_index("Method")
print()
display(arpu_results)

### 7.2 📝 Conclusion: revenue per exposed user

- **Zero-revenue users:** 87.43% in A and 84.04% in B; skewness of 3.47 and 2.85, which confirms why the bootstrap is the main method.
- **Revenue per user:** **7.68** in A vs. **10.97** in B (monetary units).
- **Difference (B − A):** **+3.29 monetary units per user**, 95% bootstrap CI **[+2.79, +3.79]**.
- **Relative change:** **+42.83%**, 95% bootstrap CI **[+35.30%, +50.79%]**; bootstrap p-value < 1e-04.
- **Welch's t-test:** t = 12.8438 (positive because B is passed first), p = 1.11e-37.

**✅ Statistical decision:** both methods agree. The difference is not compatible with chance: **Version B generates significantly more revenue per visitor**.

**🧩 Why the effect is so large:** the lift compounds two effects. B converts **26.92% more** visitors, and those buyers spend **12.5% more**: 1.2692 × 1.1254 ≈ 1.428, i.e. **+42.83% revenue per visitor**.

### 7.3 A/B scorecard

In [ ]:
# ============================================================
# 7.3 A/B SCORECARD: THE THREE BUSINESS METRICS SIDE BY SIDE
# ============================================================
def mean_ci_halfwidth(values):
    """Half-width of the normal-approximation 95% CI of a mean."""
    values = np.asarray(values, dtype=float)
    return norm.ppf(0.975) * values.std(ddof=1) / np.sqrt(len(values))

scorecard = pd.DataFrame({
    "Conversion rate (%)": [rate_A * 100, rate_B * 100],
    "Spend per buyer": [mean_spend_A, mean_spend_B],
    "Revenue per user": [arpu_A, arpu_B],
}, index=["Version A", "Version B"])

scorecard_errors = pd.DataFrame({
    "Conversion rate (%)": [norm.ppf(0.975) * np.sqrt(r * (1 - r) / n) * 100
                            for r, n in [(rate_A, n_A), (rate_B, n_B)]],
    "Spend per buyer": [mean_ci_halfwidth(spend_A), mean_ci_halfwidth(spend_B)],
    "Revenue per user": [mean_ci_halfwidth(revenue_A), mean_ci_halfwidth(revenue_B)],
}, index=scorecard.index)

lifts = {
    "Conversion rate (%)": f"{diff_rate * 100:+.2f} pp ({lift_rate:+.2f}%)",
    "Spend per buyer": f"{diff_spend:+.2f} ({lift_spend:+.1f}%)",
    "Revenue per user": f"{arpu_diff:+.2f} ({arpu_lift:+.2f}%)",
}
units = {
    "Conversion rate (%)": "%",
    "Spend per buyer": "monetary units",
    "Revenue per user": "monetary units",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for ax, metric in zip(axes, scorecard.columns):
    values = scorecard[metric]
    errors = scorecard_errors[metric]
    bars = ax.bar(scorecard.index, values, yerr=errors, capsize=8,
                  color=[COLOR_A, COLOR_B], edgecolor="black", linewidth=0.6)
    for bar, value, err in zip(bars, values, errors):
        ax.text(bar.get_x() + bar.get_width() / 2, value + err + values.max() * 0.02,
                f"{value:.2f}", ha="center", fontweight="bold")
    ax.set_title(f"{metric}\nB − A: {lifts[metric]}", fontsize=12)
    ax.set_ylabel(units[metric])
    ax.set_ylim(0, (values + errors).max() * 1.18)

fig.suptitle("A/B scorecard: Version A vs. Version B (error bars: 95% CI)",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

print("🧾 SCORECARD")
display(scorecard.round(2))

**📌 Reading the scorecard:** all three panels start at 0, so bar heights are directly comparable. Version B is ahead on every metric, and in each panel the confidence intervals of A and B do not overlap. The right-hand panel, revenue per user, is the one that drives the decision.

## 🧭 Step 8: Executive Summary for Stakeholders

### Landing page A/B test: results and recommendation

**Context.** 40,000 users of an e-commerce site were randomly split between two versions of the landing page from January 1 to 28, 2026. The two groups are comparable in size and in their mix of regions, devices, traffic sources and user types.

**Bottom line: Version B outperforms Version A on all three business metrics, and all three differences are statistically significant.**

### 1. 🏆 Version A vs. Version B

| Metric | Version A | Version B | Absolute difference (B − A) | Relative change |
|---|---:|---:|---:|---:|
| Users exposed | 19,982 | 20,018 | — | — |
| Conversions | 2,512 | 3,194 | — | — |
| Conversion rate | 12.57% | 15.96% | +3.38 percentage points | +26.92% |
| Average spend per buyer (monetary units) | 61.09 | 68.75 | +7.66 monetary units | +12.5% |
| Revenue per exposed user (monetary units) | 7.68 | 10.97 | +3.29 monetary units | +42.83% |

- **Conversion:** B converts **3.38 percentage points more** visitors than A (+26.92%), about 34 extra buyers per 1,000 visitors.
- **Spend per buyer:** buyers on B spend **7.66 monetary units more** (+12.5%), a result that holds across all robustness checks.
- **Revenue per exposed user:** B generates **3.29 monetary units more per visitor (+42.83%)**. The 95% confidence interval for the relative increase runs from **+35.30% to +50.79%**, so a plausible range for the true lift is **roughly +35% to +51%**.

The two effects compound: **more visitors buy, and those who buy spend more**.

### 2. 📡 Traffic sources

- **Organic** brings the most buyers (2,480; 43.46% of all conversions).
- **Email** has the highest observed conversion rate (14.99%), followed by Ads (14.74%), Referral (13.88%) and Organic (13.79%).
- The differences between channels are statistically detectable but **very small in practice**: only 1.2 percentage points separate the best and worst channel, and the evidence is moderate.

### 3. 👥 User types

- New users convert at 14.36% and returning users at 14.09%: **no significant difference**.
- Version B shows higher conversion rates in both segments (descriptive observation).

### 4. 💡 Recommendations

1. **Roll out Version B.** It beats A on conversion, spend per buyer and revenue per visitor, with statistical support on all three metrics.
2. **Evaluate channels on economics, not just conversion.** Organic matters for volume and Email stands out on rate, but the gaps are small. Acquisition cost and profitability per channel should be reviewed before moving budget.
3. **Do not build separate conversion strategies for new and returning users** on this evidence alone.
4. **Monitor after launch.** Track conversion rate, revenue per user and spend per buyer (ideally keeping a small holdout on Version A) to confirm that the lift persists outside the experiment.

### 5. ⚠️ Limitations

- **Randomization:** the conclusions assume users were assigned correctly. The SRM and balance checks support this, but they cannot rule out issues such as tracking errors.
- **Duration:** the test ran for 28 days. Novelty effects, seasonality and long-term outcomes (repeat purchases, returns, refunds) are not captured.
- **User type:** the advice not to segment by user type rests on the overall chi-square test and descriptive per-version rates, not on a formal page × user-type interaction test.
- **Channels:** only a global test was run (no pairwise comparisons), and there is no cost data per channel, so no conclusions about channel ROI can be drawn.
- **Currency:** spend is reported in unspecified monetary units.

### 🎯 Final recommendation

**Roll out Version B.** It delivers 26.92% more conversions and 42.83% more revenue per visitor, with a plausible range of +35% to +51% for the revenue lift. Differences by channel and user type are small or not significant, so the decision should rest on the overall performance of the two versions.

*The statistical details behind each figure are in Steps 1–7.*

## 💾 Step 9: Export the Clean Dataset

The validated, English-standardized dataset is exported as `landing_experiment_clean.csv`. This makes the project:

- **Reproducible:** anyone can rerun the analysis from the same clean file.
- **Reusable:** the file can be loaded directly into Power BI, Tableau, Excel or SQL.
- **Traceable:** the export only happens after every quality rule passes again, so a broken file is never written.

In [ ]:
# ============================================================
# 9. EXPORT THE CLEAN DATASET
# ============================================================
df_clean = df.copy()
df_clean["date"] = df_clean["date"].dt.strftime("%Y-%m-%d")

# Re-check every quality rule before writing the file
assert not df_clean.isna().any().any(), "There are missing values"
assert df_clean["user_id"].is_unique, "There are repeated user IDs"
assert df_clean["landing"].isin(["A", "B"]).all(), "Unexpected landing values"
assert df_clean["converted"].isin([0, 1]).all(), "converted must be 0 or 1"
assert (df_clean["spend"] >= 0).all(), "There are negative spend values"
assert (df_clean.loc[df_clean["converted"] == 0, "spend"] == 0).all(), "Non-buyers with spend"
assert (df_clean.loc[df_clean["converted"] == 1, "spend"] > 0).all(), "Buyers without spend"

CLEAN_FILE = "landing_experiment_clean.csv"
df_clean.to_csv(CLEAN_FILE, index=False, encoding="utf-8")

# Read the file back to confirm it was written correctly
df_check = pd.read_csv(CLEAN_FILE)
assert df_check.shape == df_clean.shape, "The exported file has a different shape"
assert list(df_check.columns) == list(df_clean.columns), "The exported columns differ"
assert df_check["converted"].sum() == df_clean["converted"].sum(), "Conversions do not match"
assert np.isclose(df_check["spend"].sum(), df_clean["spend"].sum()), "Total spend does not match"

print("💾 CLEAN DATASET EXPORTED")
print("-" * 55)
print(f"📄 File: {CLEAN_FILE}")
print(f"📐 Shape: {df_check.shape[0]:,} rows × {df_check.shape[1]} columns")
print(f"📋 Columns: {list(df_check.columns)}")
print("✅ Read-back check passed")
display(df_check.head())

# In Google Colab, download the file automatically
try:
    from google.colab import files
    files.download(CLEAN_FILE)
    print("⬇️ Download started (check your browser's downloads)")
except ImportError:
    print(f"📁 Saved next to the notebook: {Path(CLEAN_FILE).resolve()}")

---

### 🚀 Possible next steps

- Run a formal **page × user type interaction test** (e.g., logistic regression) to confirm that B's lift is the same for new and returning users.
- Add **pairwise channel comparisons** with a multiple-testing correction, combined with acquisition cost data, to evaluate channel ROI.
- Keep monitoring Version B after launch to confirm the lift holds beyond the 28-day test window.